Set up SQLite database

In [6]:
import sqlite3
import pandas as pd
df = pd.read_csv('hr_attrition_clean.csv')

conn = sqlite3.connect('hr_attrition.db')
df.to_sql('employees', conn, if_exists='replace', index=False)
print("Database Created!")
print(f"Total records loaded: {len(df)}")

Database Created!
Total records loaded: 1470


Helper function

In [7]:
def run_query(sql, title=""):
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f"\n{'='*50}")
        print(f"  {title}")
        print('='*50)
    print(result.to_string(index=False))
    return result

Overall attrition count and rate

In [8]:
run_query("""
SELECT
     Attrition,
     COUNT(*) AS employee_count,
     ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM employees), 1) AS percentage
FROM employees
GROUP BY Attrition
ORDER BY Attrition DESC
""", "Q1: Overall Attrition Rate")


  Q1: Overall Attrition Rate
Attrition  employee_count  percentage
      Yes             237        16.1
       No            1233        83.9


,Attrition,employee_count,percentage
0,Yes,237,16.1
1,No,1233,83.9


Attrition by Department

In [9]:
run_query("""
SELECT
    Department,
    COUNT(*) AS total_employees,
    SUM(Attrition_Flag) AS attrited,
    COUNT(*) - SUM(Attrition_Flag) AS retained,
    ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY Department
ORDER BY attrition_rate_pct DESC
""", "Q2: Attrition by Department")


  Q2: Attrition by Department
            Department  total_employees  attrited  retained  attrition_rate_pct
                 Sales              446        92       354                20.6
       Human Resources               63        12        51                19.0
Research & Development              961       133       828                13.8


,Department,total_employees,attrited,retained,attrition_rate_pct
0,Sales,446,92,354,20.6
1,Human Resources,63,12,51,19.0
2,Research & Development,961,133,828,13.8


Overtime impact

In [10]:
run_query("""
SELECT
      OverTime,
      COUNT(*) AS total,
      SUM(Attrition_Flag) AS attrited,
      ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct,
      ROUND(AVG(MonthlyIncome), 0) AS avg_income
FROM employees
GROUP BY OverTime
""", "Q3: Overtime vs No Overtime")


  Q3: Overtime vs No Overtime
OverTime  total  attrited  attrition_rate_pct  avg_income
      No   1054       110                10.4      6485.0
     Yes    416       127                30.5      6549.0


,OverTime,total,attrited,attrition_rate_pct,avg_income
0,No,1054,110,10.4,6485.0
1,Yes,416,127,30.5,6549.0


 Average income comparison

In [11]:
run_query("""
SELECT
      Attrition,
      ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income,
      ROUND(AVG(YearsAtCompany), 1) AS avg_years_at_company,
      ROUND(AVG(Age), 1) AS avg_age,
      COUNT(*) AS count
FROM employees
GROUP BY Attrition
""", "Q4: Income and Tenure Comparison")
      


  Q4: Income and Tenure Comparison
Attrition  avg_monthly_income  avg_years_at_company  avg_age  count
       No              6833.0                   7.4     37.6   1233
      Yes              4787.0                   5.1     33.6    237


,Attrition,avg_monthly_income,avg_years_at_company,avg_age,count
0,No,6833.0,7.4,37.6,1233
1,Yes,4787.0,5.1,33.6,237


Top 5 job roles by attrition

In [12]:
run_query("""
SELECT 
      JobRole,
      COUNT(*) AS total,
      SUM(Attrition_Flag) as attrited,
      ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY JobRole
ORDER BY attrition_rate_pct DESC
LIMIT 5
""", "Q5: Top 5 job roles by attrition")


  Q5: Top 5 job roles by attrition
              JobRole  total  attrited  attrition_rate_pct
 Sales Representative     83        33                39.8
Laboratory Technician    259        62                23.9
      Human Resources     52        12                23.1
      Sales Executive    326        57                17.5
   Research Scientist    292        47                16.1


,JobRole,total,attrited,attrition_rate_pct
0,Sales Representative,83,33,39.8
1,Laboratory Technician,259,62,23.9
2,Human Resources,52,12,23.1
3,Sales Executive,326,57,17.5
4,Research Scientist,292,47,16.1


Age group analysis

In [13]:
run_query("""
SELECT
    CASE
        WHEN Age < 25 THEN 'Under 25'
        WHEN Age BETWEEN 25 AND 34 THEN '25–34'
        WHEN Age BETWEEN 35 AND 44 THEN '35–44'
        WHEN Age BETWEEN 45 AND 54 THEN '45–54'
        ELSE '55 and above'
    END AS age_group,
    COUNT(*) AS total,
    SUM(Attrition_Flag) AS attrited,
    ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY age_group
ORDER BY attrition_rate_pct DESC
""", "Q6: Attrition by Age Group")


  Q6: Attrition by Age Group
   age_group  total  attrited  attrition_rate_pct
    Under 25     97        38                39.2
       25–34    554       112                20.2
55 and above     69        11                15.9
       45–54    245        25                10.2
       35–44    505        51                10.1


,age_group,total,attrited,attrition_rate_pct
0,Under 25,97,38,39.2
1,25–34,554,112,20.2
2,55 and above,69,11,15.9
3,45–54,245,25,10.2
4,35–44,505,51,10.1


Tenure group analysis

In [14]:
run_query("""
SELECT
    CASE
        WHEN YearsAtCompany <= 2  THEN '0–2 years'
        WHEN YearsAtCompany <= 5  THEN '3–5 years'
        WHEN YearsAtCompany <= 10 THEN '6–10 years'
        WHEN YearsAtCompany <= 20 THEN '11–20 years'
        ELSE 'Over 20 years'
    END AS tenure_group,
    COUNT(*) AS total,
    ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct,
    ROUND(AVG(MonthlyIncome), 0) AS avg_income
FROM employees
GROUP BY tenure_group
ORDER BY attrition_rate_pct DESC
""", "Q7: Attrition by Tenure Group")


  Q7: Attrition by Tenure Group
 tenure_group  total  attrition_rate_pct  avg_income
    0–2 years    342                29.8      4709.0
    3–5 years    434                13.8      5365.0
   6–10 years    448                12.3      6567.0
Over 20 years     66                12.1     16008.0
  11–20 years    180                 6.7      9009.0


,tenure_group,total,attrition_rate_pct,avg_income
0,0–2 years,342,29.8,4709.0
1,3–5 years,434,13.8,5365.0
2,6–10 years,448,12.3,6567.0
3,Over 20 years,66,12.1,16008.0
4,11–20 years,180,6.7,9009.0


High risk segment (overtime + low satisfaction)

In [15]:
run_query("""
SELECT
    OverTime,
    JobSatisfaction,
    COUNT(*) AS total,
    SUM(Attrition_Flag) AS attrited,
    ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY OverTime, JobSatisfaction
ORDER BY attrition_rate_pct DESC
LIMIT 8
""", "Q8: High Risk Combinations (OverTime + Satisfaction)")


  Q8: High Risk Combinations (OverTime + Satisfaction)
OverTime  JobSatisfaction  total  attrited  attrition_rate_pct
     Yes                2     69        26                37.7
     Yes                1     84        30                35.7
     Yes                3    121        41                33.9
     Yes                4    142        30                21.1
      No                1    205        36                17.6
      No                3    321        32                10.0
      No                2    211        20                 9.5
      No                4    317        22                 6.9


,OverTime,JobSatisfaction,total,attrited,attrition_rate_pct
0,Yes,2,69,26,37.7
1,Yes,1,84,30,35.7
2,Yes,3,121,41,33.9
3,Yes,4,142,30,21.1
4,No,1,205,36,17.6
5,No,3,321,32,10.0
6,No,2,211,20,9.5
7,No,4,317,22,6.9


Income bracket analysis

In [16]:
run_query("""
SELECT
    CASE
        WHEN MonthlyIncome < 3000  THEN 'Under $3K'
        WHEN MonthlyIncome < 6000  THEN '$3K–6K'
        WHEN MonthlyIncome < 10000 THEN '$6K–10K'
        ELSE 'Above $10K'
    END AS income_bracket,
    COUNT(*) AS total,
    ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY income_bracket
ORDER BY attrition_rate_pct DESC
""", "Q9: Attrition by Income Bracket")


  Q9: Attrition by Income Bracket
income_bracket  total  attrition_rate_pct
     Under $3K    395                28.6
        $3K–6K    519                12.7
       $6K–10K    275                12.0
    Above $10K    281                 8.9


,income_bracket,total,attrition_rate_pct
0,Under $3K,395,28.6
1,$3K–6K,519,12.7
2,$6K–10K,275,12.0
3,Above $10K,281,8.9


Work-life balance analysis

In [17]:
run_query("""
SELECT
    WorkLifeBalance,
    CASE WorkLifeBalance
        WHEN 1 THEN '1-Bad'
        WHEN 2 THEN '2-Good'
        WHEN 3 THEN '3-Better'
        WHEN 4 THEN '4-Best'
    END AS balance_label,
    COUNT(*) AS total,
    ROUND(AVG(Attrition_Flag) * 100, 1) AS attrition_rate_pct
FROM employees
GROUP BY WorkLifeBalance
ORDER BY WorkLifeBalance
""", "Q10: Work-Life Balance vs Attrition")

conn.close()
print("\nAll 10 queries completed. Database connection closed.")


  Q10: Work-Life Balance vs Attrition
 WorkLifeBalance balance_label  total  attrition_rate_pct
               1         1-Bad     80                31.3
               2        2-Good    344                16.9
               3      3-Better    893                14.2
               4        4-Best    153                17.6

All 10 queries completed. Database connection closed.
